In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

property = pd.read_csv("../data/datasets/property.csv")
rent = pd.read_csv("../data/datasets/rent.csv")

# Filtering only "NM"
property_nm = property[property["State"]=="NM"]
rent_nm = rent[rent["State"]=="NM"]

print(property_nm.head)
print(rent_nm.shape)

In [ ]:
# clean the dataset [zip, price], [zip, rent]
latest_price = property_nm.columns[-1]
latest_rent = rent_nm.columns[-1]

property_clean = property_nm[["RegionName", latest_price]]
rent_clean = rent_nm[["RegionName", latest_rent]]

property_clean.columns = ["zip", "price"]
rent_clean.columns = ["zip", "rent"]

In [ ]:
# merge two dataset by zip
df = pd.merge(property_clean, rent_clean, on='zip')
# change data type of zip from float to str
df["zip"] = df["zip"].astype(str)
df

In [ ]:
# calcualte ROI
df["ROI"] = (df["rent"]*12) / df["price"] * 100
df

In [ ]:
income = pd.read_csv("../data/datasets/income.csv")

# Clear median income data [zip, median_income]
income = income.iloc[1:]
income = income[["NAME", "B19013_001E"]]
income.columns = ["zip", "median_income"]
income["zip"] = income["zip"].str[-5:]
income_nm = income[income["zip"].isin(df["zip"])]

income

In [ ]:
# [zip, price, rent, ROI, median_price]
investment_df = pd.merge(df, income, on="zip", how="left")
investment_df

In [ ]:
# highest income
investment_df.sort_values("median_income", ascending=False).head()

In [ ]:
# highest ROI
investment_df.sort_values("ROI", ascending=False).head()

In [ ]:
# Relation ROI vs income
investment_df.plot.scatter(x="median_income", y="ROI")
investment_df.dtypes

In [ ]:
investment_df["median_income"] = pd.to_numeric(investment_df["median_income"])
investment_df[
    (investment_df["ROI"] > 0.08) &
    (investment_df["median_income"] > 50000)
].sort_values("ROI", ascending=False)

In [ ]:
mortgage_rate = 0.065
investment_df["annual_mortgage_interest_cost"] = investment_df["price"]*mortgage_rate
investment_df[["zip","price","annual_mortgage_interest_cost"]].head()

In [ ]:
investment_df["cash_flow"] = (
    investment_df["rent"]*12
  - investment_df["annual_mortgage_interest_cost"]
)
investment_df[["zip","annual_mortgage_interest_cost","cash_flow"]].head()

In [ ]:
investment_df["ROI_after_interest"] = (
    investment_df["cash_flow"] / investment_df["price"]
)

investment_df[["zip","ROI","ROI_after_interest"]].head()

In [ ]:
def z(s):
    return (s - s.mean()) / s.std(ddof=0)

# Standardized column
investment_df["z_roi_ai"] = z(investment_df["ROI_after_interest"])
investment_df["z_income"] = z(investment_df["median_income"])
investment_df["z_price"]  = z(investment_df["price"])

# 3) Investment Score (가중치: ROI 0.5, Income 0.3, Price(낮을수록 좋음) 0.2)
investment_df["investment_score"] = (
    0.5 * investment_df["z_roi_ai"]
    + 0.3 * investment_df["z_income"]
    - 0.2 * investment_df["z_price"]
)

investment_df[["zip","ROI_after_interest","median_income","price","investment_score"]].head()

In [ ]:
top10 = investment_df.sort_values(
    "investment_score", ascending=False
).head(10)

top10[["zip","price","rent","ROI_after_interest","median_income","investment_score"]]

In [ ]:
investment_df.plot.scatter(
    x="median_income",
    y="investment_score"
)

In [ ]:
investment_df.to_csv("newmexico_real_estate_investment.csv", index=False)

In [ ]:
newmexico_df = investment_df